# EVT Analysis — GEV (Full) & GPD (Tail) with Trim
This notebook mirrors the trimming and EVT fitting logic in your `MyAnalysis.py`:
- **Trim** by removing the **top p%** of the values (like `trim_top_pct`).
- **GEV (full data)**: fit on the untrimmed data and report KS test + AIC/BIC.
- **Best trim**: choose the trim level with the **highest KS p-value** (GEV on trimmed data).
- **Threshold `u`**: set from the best trim via `q = 1 - best_trim`, clamped to `[0.80, 0.995]` (fallback 0.95).
- **GPD (tail)**: fit on exceedances `X - u` (with `loc=0`), report KS test + AIC/BIC.
- **Plots**: histogram + GEV PDF (full data) and QQ-plot for GPD tail, **saved per metric**.
- **Scope**: only `producer_application` and `consumer_application` metrics.


In [4]:
# =====================
# Config
# =====================
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional, Tuple, Dict

from scipy.stats import genextreme, genpareto, kstest

# ---- INPUTS ----
# Path to a block maxima CSV that has (at least) columns:
#   ["metric", "block_max_value"] and optionally ["target_rate"]
# You can generate this from your pipeline, or reuse an existing one.
Home = "/Users/soheila/Desktop/RealTime-Streaming-Pipeline/resuls"
block_path = os.path.join(Home, "20250908_140639/merge/merge-sts-0_merge-metrics/2025-09-08_00-57-10/block_maxima.csv")  # <-- EDIT IF NEEDED

# Metrics to analyze (subset)
METRICS = ("producer_application", "consumer_application")

# Trim levels (top-p% removal) to consider for "best trim" by KS p-value
TRIM_LEVELS = (0.00, 0.01, 0.02, 0.05, 0.07, 0.10)

# Minimum exceedances needed to fit GPD
MIN_EXCEEDANCES = 50

# ---- OUTPUT ----
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
PLOTS_ROOT = Path(Home) / "results" / "evt" / f"run_{RUN_STAMP}"
PLOTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f"[Init] Output directory: {PLOTS_ROOT}")


[Init] Output directory: /Users/soheila/Desktop/RealTime-Streaming-Pipeline/resuls/results/evt/run_20250909_153328


In [5]:
# =====================
# Helpers (trim, fits, plots)
# =====================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional, Tuple, Dict
from scipy.stats import genextreme, genpareto, kstest

def trim_top_pct(arr: np.ndarray, pct: float) -> np.ndarray:
    """
    Remove the top 'pct' fraction (e.g., 0.05 -> remove top 5%).
    Same behavior as in MyAnalysis.py.
    """
    arr = np.asarray(arr, float)
    arr = arr[np.isfinite(arr)]
    if pct <= 0:
        return arr
    cut = np.nanpercentile(arr, 100.0 * (1.0 - pct))
    return arr[arr <= cut]

def aic_bic_from_logpdf(logpdf: np.ndarray, k_params: int) -> Tuple[float, float]:
    """Compute AIC/BIC from a vector of log-pdf values and number of parameters."""
    ll = float(np.nansum(logpdf))
    n  = int(np.sum(np.isfinite(logpdf)))
    aic = 2 * k_params - 2 * ll
    bic = k_params * np.log(max(n, 1)) - 2 * ll
    return float(aic), float(bic)

def fit_gev_full(x: np.ndarray) -> Dict[str, float]:
    """GEV fit on full data (no trim), with KS and AIC/BIC (as in MyAnalysis.py)."""
    xx = np.asarray(x, float)
    c, loc, scale = genextreme.fit(xx)
    D, p = kstest(xx, "genextreme", args=(c, loc, scale))
    logpdf = genextreme.logpdf(xx, c, loc=loc, scale=scale)
    aic, bic = aic_bic_from_logpdf(logpdf, k_params=3)
    return dict(c=c, loc=loc, scale=scale, KS_D=D, KS_p=p, AIC=aic, BIC=bic)

def best_trim_by_ks(x: np.ndarray, trim_levels=()) -> Tuple[float, Dict[str, float]]:
    """
    Return (best_trim, params_dict) where 'best_trim' is the trim level that maximizes KS p-value
    for the GEV fit on trimmed data, mirroring your logic.
    """
    best_trim = 0.0
    best = None
    for pct in trim_levels:
        xt = trim_top_pct(x, pct)
        if xt.size < 10:
            continue
        c, loc, scale = genextreme.fit(xt)
        D, p = kstest(xt, "genextreme", args=(c, loc, scale))
        if (best is None) or (p > best["KS_p"]):
            best = dict(c=c, loc=loc, scale=scale, KS_D=D, KS_p=p, n=len(xt), trim=pct)
            best_trim = pct
    return best_trim, (best or {})

def threshold_from_best_trim(x: np.ndarray, best_trim: float, fallback_q: float = 0.95) -> float:
    """
    Convert best trim into a quantile threshold u:
        q = 1 - best_trim, clamped to [0.80, 0.995].
    If best_trim <= 0, fall back to q=0.95.
    """
    if best_trim is None or best_trim <= 0.0:
        q = fallback_q
    else:
        q = 1.0 - best_trim
        q = min(max(q, 0.80), 0.995)
    return float(np.nanpercentile(np.asarray(x, float), q * 100.0))

def fit_gpd_tail(x: np.ndarray, u: float, min_exc: int = 50) -> Optional[Dict[str, float]]:
    """
    Fit GPD to exceedances above threshold u (exc = x[x>u] - u).
    MLE fit with loc fixed to 0; report KS and AIC/BIC for the tail model only.
    """
    xx = np.asarray(x, float)
    exc = xx[xx > u] - u
    exc = exc[np.isfinite(exc)]
    if exc.size < min_exc:
        return None
    shape, loc, scale = genpareto.fit(exc, floc=0)
    D, p = kstest(exc, "genpareto", args=(shape, 0, scale))
    logpdf = genpareto.logpdf(exc, shape, 0, scale)
    aic, bic = aic_bic_from_logpdf(logpdf, k_params=2)  # shape, scale
    return dict(xi=shape, beta=scale, KS_D=D, KS_p=p, n_exc=len(exc), u=u)

def plot_hist_with_gev(x: np.ndarray, params: Dict[str, float], title: str, out_path: Path):
    """Histogram + fitted GEV PDF (single figure)."""
    import matplotlib.pyplot as plt
    xx = np.linspace(float(np.min(x)), float(np.max(x)), 400)
    pdf = genextreme.pdf(xx, params["c"], loc=params["loc"], scale=params["scale"])

    plt.figure()
    plt.hist(x, bins=40, density=True, alpha=0.6, label="Empirical")
    plt.plot(xx, pdf, label="GEV fit")
    plt.title(title)
    plt.xlabel("Block maxima")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()

def plot_gpd_tail_qq(x: np.ndarray, gpd_params: Optional[Dict[str, float]], title: str, out_path: Path):
    """QQ plot for GPD exceedances (single figure)."""
    import matplotlib.pyplot as plt
    plt.figure()
    if gpd_params is None:
        plt.text(0.5, 0.5, "Not enough exceedances for GPD", ha="center", va="center")
        plt.title(title)
        plt.xticks([]); plt.yticks([])
    else:
        u = gpd_params["u"]
        exc = x[x > u] - u
        exc = np.sort(exc)
        if exc.size < 2:
            plt.text(0.5, 0.5, "Not enough exceedances for GPD", ha="center", va="center")
            plt.title(title)
            plt.xticks([]); plt.yticks([])
        else:
            q = np.linspace(0.01, 0.99, len(exc))
            theo = genpareto.ppf(q, gpd_params["xi"], 0, gpd_params["beta"])
            plt.scatter(theo, exc, s=10, alpha=0.6)
            lo, hi = float(np.nanmin(theo)), float(np.nanmax(theo))
            plt.plot([lo, hi], [lo, hi])
            plt.title(title + f"  (KS p={gpd_params['KS_p']:.3f}, n_exc={gpd_params['n_exc']})")
            plt.xlabel("Theoretical (GPD)")
            plt.ylabel("Exceedances")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


In [6]:
# =====================
# Load data and run EVT analysis
# =====================
import os
import numpy as np
import pandas as pd

# Load block maxima
if not os.path.exists(block_path):
    raise FileNotFoundError(f"block_maxima.csv not found at: {block_path}")

block_df = pd.read_csv(block_path)

# Basic cleaning / clipping — mirror common practice in your scripts
# (You can change these bounds if your units are ms vs s)
if "block_max_value" not in block_df.columns:
    raise ValueError("block_maxima.csv must include 'block_max_value'.")

# Keep sensible range (avoid negatives, overly large outliers)
block_df = block_df[(block_df["block_max_value"] >= 0) & (block_df["block_max_value"] <= 5.0)].copy()

# Ensure required columns are strings/numeric as needed
block_df["metric"] = block_df["metric"].astype(str)

# Where to save plots per metric
for m in METRICS:
    (PLOTS_ROOT / m).mkdir(parents=True, exist_ok=True)

summary_rows = []

for metric in METRICS:
    sub = block_df[block_df["metric"] == metric].copy()
    if sub.empty:
        print(f"[Skip] No rows for metric={metric}")
        continue

    # If target_rate exists, analyze per target_rate; else do all at once with TR=None
    tr_values = sorted(sub["target_rate"].dropna().unique()) if "target_rate" in sub.columns else [None]

    for tr in tr_values:
        if tr is None:
            data = sub["block_max_value"].dropna().astype(float).values
        else:
            data = sub.loc[sub["target_rate"] == tr, "block_max_value"].dropna().astype(float).values

        data = data[np.isfinite(data)]
        if data.size < 20:
            print(f"[Skip] metric={metric}, TR={tr} has too few samples (n={data.size})")
            continue

        # GEV (full data)
        gev_params = fit_gev_full(data)

        # Best trim by KS
        best_trim, best_trim_params = best_trim_by_ks(data, trim_levels=TRIM_LEVELS)

        # Threshold from best trim (fallback 0.95)
        u = threshold_from_best_trim(data, best_trim, fallback_q=0.95)

        # GPD tail on exceedances above u
        gpd_params = fit_gpd_tail(data, u=u, min_exc=MIN_EXCEEDANCES)

        # --- Plot and save
        base = f"{metric}_tr{int(tr) if tr is not None else 'ALL'}"

        # Histogram + GEV PDF
        out1 = (PLOTS_ROOT / metric / f"hist_gev__{base}.png")
        plot_hist_with_gev(data, gev_params, title=f"{metric} | TR={tr} — Full-data GEV", out_path=out1)

        # Tail QQ for GPD
        out2 = (PLOTS_ROOT / metric / f"qq_gpd__{base}.png")
        plot_gpd_tail_qq(data, gpd_params, title=f"{metric} | TR={tr} — Tail QQ (GPD over u={u:.4g})", out_path=out2)

        # Collect summary
        summary_rows.append({
            "metric": metric,
            "target_rate": (None if tr is None else int(tr)),
            "n": int(data.size),
            "GEV_c": float(gev_params["c"]),
            "GEV_loc": float(gev_params["loc"]),
            "GEV_scale": float(gev_params["scale"]),
            "GEV_KS_D": float(gev_params["KS_D"]),
            "GEV_KS_p": float(gev_params["KS_p"]),
            "best_trim": float(best_trim or 0.0),
            "u": float(u),
            "GPD_xi": (np.nan if gpd_params is None else float(gpd_params["xi"])),
            "GPD_beta": (np.nan if gpd_params is None else float(gpd_params["beta"])),
            "GPD_KS_D": (np.nan if gpd_params is None else float(gpd_params["KS_D"])),
            "GPD_KS_p": (np.nan if gpd_params is None else float(gpd_params["KS_p"])),
            "GPD_n_exc": (np.nan if gpd_params is None else int(gpd_params["n_exc"])),
            "hist_gev_png": str(out1),
            "qq_gpd_png": str(out2),
        })

# Save summary CSV
summary_df = pd.DataFrame(summary_rows)
csv_out = PLOTS_ROOT / "evt_summary.csv"
summary_df.to_csv(csv_out, index=False)
print(f"[Done] Summary written to: {csv_out}")


FileNotFoundError: block_maxima.csv not found at: /Users/soheila/Desktop/RealTime-Streaming-Pipeline/resuls/20250908_140639/merge/merge-sts-0_merge-metrics/2025-09-08_00-57-10/block_maxima.csv

In [7]:
# =====================
# Zip all outputs (optional)
# =====================
import os, zipfile
zip_path = PLOTS_ROOT.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(PLOTS_ROOT):
        for f in files:
            fp = Path(root) / f
            arcname = fp.relative_to(PLOTS_ROOT.parent)
            zf.write(fp, arcname)
print(f"[Zip] {zip_path}")


[Zip] /Users/soheila/Desktop/RealTime-Streaming-Pipeline/resuls/results/evt/run_20250909_153328.zip
